# Ignite-3B S09 - Ignition test (L2 gate)

**Key novel result**: does v_N as outer produce better v_1 than v_0 as outer?

Protocol:
1. Load v_0 (base) and v_N (final from Cond C)
2. Both propose N=8 mutations on same fresh state
3. Train candidates from both proposers
4. McNemar paired test on held-out

**L2 confirmed** if p<0.01 AND sigmoid asymptote CI>0.

In [ ]:
BASE = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
V0_REPO = None  # base model, no adapter
VN_REPO = 'iterate-labs-ai/ignite-3b-v1'
OUT = '/kaggle/working/ignition'
CANDS = 8
STEPS = 100

In [ ]:
!pip install -q -U 'transformers>=4.46.0' 'peft>=0.13.0' 'datasets>=3.0.0' 'accelerate>=1.0.0' 'unsloth>=2025.1.0' 'trl>=0.12.0' 'vllm>=0.6.0' 'math-verify>=0.5.2' 'latex2sympy2' 'sympy' 'scipy' 'huggingface_hub'
import os, subprocess
if not os.path.exists('/kaggle/working/caracal-1'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 's07-hybrid-agentic',
                    'https://github.com/iterate-labs-ai/caracal-1.git',
                    '/kaggle/working/caracal-1'], check=True)
os.chdir('/kaggle/working/caracal-1')
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret('HF_TOKEN'))

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download
import subprocess, sys, os

os.makedirs(OUT, exist_ok=True)
vn_adapter = snapshot_download(repo_id=VN_REPO)
print(f'v_N adapter -> {vn_adapter}')

# Run outer_loop 1 gen with v_0 as proposer (adapter=None)
subprocess.run([sys.executable, '-m', 'train.ignite.C_rsi_outer',
       '--base', BASE,
       '--dataset-train', 'data/ignite/omni_math_train.jsonl',
       '--dataset-dev', 'data/ignite/omni_math_dev.jsonl',
       '--dataset-val', 'data/ignite/omni_math_val.jsonl',
       '--bench', 'math', '--bench-name', 'omni_math',
       '--gens', '1', '--cands', str(CANDS), '--steps', str(STEPS),
       '--out', f'{OUT}/v0_as_outer'], check=True)

# Run outer_loop 1 gen with v_N as proposer
subprocess.run([sys.executable, '-m', 'train.ignite.C_rsi_outer',
       '--base', BASE, '--v0-adapter', vn_adapter,
       '--dataset-train', 'data/ignite/omni_math_train.jsonl',
       '--dataset-dev', 'data/ignite/omni_math_dev.jsonl',
       '--dataset-val', 'data/ignite/omni_math_val.jsonl',
       '--bench', 'math', '--bench-name', 'omni_math',
       '--gens', '1', '--cands', str(CANDS), '--steps', str(STEPS),
       '--out', f'{OUT}/vN_as_outer'], check=True)

In [ ]:
import json
from eval.ignite.stats import mcnemar_test, sigmoid_fit

def best_val_per_sample(run_dir):
    log = [json.loads(l) for l in open(f'{run_dir}/log.jsonl')]
    cands = [r for r in log if r.get('type') == 'candidate' and r.get('val_r') is not None]
    if not cands: return []
    best = max(cands, key=lambda x: x['val_r'])
    return best

v0_best = best_val_per_sample(f'{OUT}/v0_as_outer')
vN_best = best_val_per_sample(f'{OUT}/vN_as_outer')
print(f'v_0-as-outer best val_r: {v0_best.get("val_r")}')
print(f'v_N-as-outer best val_r: {vN_best.get("val_r")}')

# TODO: per-sample McNemar requires re-eval both best adapters on same held-out slice
print('\n[TODO] Run eval.ignite.run_all_ignite --adapter both, then mcnemar_test(per_sample)')